# **CARE (3D) — powered by CAREamics**

---

<font size = 4> CARE (Content-Aware image REstoration) is a **supervised** deep-learning method that restores low-quality microscopy images. It is trained on **pairs of low-SNR images and matching high-SNR (ground truth) images of the same field of view**, and learns the mapping from one to the other. It was originally published by [Weigert *et al.*, Nature Methods (2018)](https://doi.org/10.1038/s41592-018-0216-7).

<font size = 4> **This notebook runs CARE on 3D datasets using [CAREamics](https://careamics.github.io/), a modern PyTorch/Lightning implementation. Your images must be 3D stacks with axes ordered `ZYX`. For 2D datasets, use the **CAREamics** CARE 2D notebook instead. This notebook is an alternative to the existing CSBDeep-based `CARE_3D_ZeroCostDL4Mic` notebook, which is unaffected by this one.**

<font size = 4> **If you do not have high-SNR ground truth**, CARE cannot be used. Consider the **CAREamics Noise2Noise** notebook (two independent noisy acquisitions) or the **CAREamics Noise2Void** notebook (noisy images alone) instead.

---

<font size = 4>*Disclaimer*:

<font size = 4>This notebook is part of the Zero-Cost Deep-Learning to Enhance Microscopy project (https://github.com/HenriquesLab/DeepLearning_Collab/wiki). Jointly developed by the Jacquemet (https://cellmig.org/) and Henriques (https://henriqueslab.github.io/) laboratories.

<font size = 4>The deep-learning engine used here is **CAREamics** (https://github.com/CAREamics/careamics).

<font size = 4>This notebook is based on:

<font size = 4>**Content-aware image restoration: pushing the limits of fluorescence microscopy**, Weigert *et al.*, Nature Methods 2018 (https://doi.org/10.1038/s41592-018-0216-7)

<font size = 4>**Please cite the original CARE paper and CAREamics when using this notebook.**


# **How to use this notebook?**

---

<font size = 4>This notebook is structured in numbered sections. Run the cells from top to bottom.

---
### **Structure of a notebook**

<font size = 4>**Text cells** provide information. **Code cells** contain code; move your cursor over the `[ ]` on the left and click the play button to execute.

---
### **Making changes to the notebook**

<font size = 4>**Make a copy** of this notebook and save it to your Google Drive (`File -> Save a copy in Drive`) before editing.

# **0. Before getting started**
---

<font size = 4>Before running the notebook, make sure you are logged into your Google account and that your data is in your Google Drive.

<font size = 4>CARE is **supervised** and requires **paired training data**: for each field of view you need a **low-SNR stack** and a matching **high-SNR stack** of the same volume. The low-SNR stack is the input, the high-SNR stack is the target the network learns to produce. Unlike Noise2Noise, the target must be genuinely higher quality — a second equally-noisy acquisition will not work.

<font size = 4>Every image must be a **3D stack** whose axes are ordered `ZYX` (slices, height, width). Stacks with a different axis order must be reordered before you start, for example with Fiji's `Image -> Hyperstacks -> Re-order Hyperstack`. If your data is 2D, use the **CAREamics** CARE 2D notebook instead.

<font size = 4>Please note that you can **only use .tif or .tiff files!**

<font size = 4>The input and target images must be provided in **two separate folders**, and paired images must have **matching file names**. Information on how to prepare a training dataset is available on the [ZeroCostDL4Mic wiki](https://github.com/HenriquesLab/ZeroCostDL4Mic/wiki).

<font size = 4>**We strongly recommend also preparing a Quality Control dataset**: a few low-SNR stacks together with matching high-SNR versions of the same volumes. These let section 5.2 measure how much the model actually improved your images. Paired files must have **matching file names**. Quality control data is the same kind of data as the training data, so **hold these fields of view back from training** — otherwise section 5.2 measures how well the model fits data it has already seen.

<font size = 4>A common data structure that works well:

*   Data
    - **Training**
        - source (low SNR) — img_1.tif, img_2.tif ...
        - target (high SNR, same scenes) — img_1.tif, img_2.tif ...
    - **Quality control** (optional but recommended)
        - Low SNR images — img_1.tif, img_2.tif ...
        - High SNR images — img_1.tif, img_2.tif ...
    - **Prediction** — images to denoise
    - **Results**

---
<font size = 4>**Important note**

<font size = 4>- To **train from scratch**: run **sections 1-4**, then **section 5** to assess quality and **section 6** to predict.
<font size = 4>- To **continue training from a checkpoint**: run **sections 1-4** with `Use_pretrained_model` enabled and training paths filled in.
<font size = 4>- To only **run predictions with an existing checkpoint**: run **sections 1-3**, enable `Use_pretrained_model`, run **section 4.1** to load the model, skip **section 4.2**, then run **section 6**.
---


# **1. Install CAREamics and dependencies**
---

## **1.1. Install CAREamics**

In [ ]:
#@markdown ##Install CAREamics and dependencies
#@markdown This installs a pinned, tested version of CAREamics. It takes several minutes,
#@markdown because CAREamics 0.3.2 requires torch < 2.10 and Colab ships a newer one, so
#@markdown PyTorch and its CUDA libraries are reinstalled.

# NumPy is deliberately NOT pinned. CAREamics 0.3.2 accepts numpy<=2.4.6, which the
# version Colab already ships satisfies, so pip leaves it alone. Pinning it lower would
# force pip to build NumPy from source on Colab's Python 3.13, which is slow and fragile.
!pip install "careamics==0.3.2" "careamics-portfolio" -q

print("CAREamics installed.")

## **1.2. Restart the runtime (only if you see an import error)**
<font size = 4>The install above keeps Colab's existing NumPy, so you can normally continue straight to section 1.3. **If section 1.3 raises a NumPy or import error**, go to `Runtime -> Restart session`, then re-run from section 1.3 (do **not** re-run the install cell).

## **1.3. Load key dependencies**

In [ ]:
#@markdown ##Load key dependencies
import csv
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tifffile

import careamics
from careamics import CAREamist
# The advanced factory is the plain one plus more options; it is the only one that
# exposes `seed`. Passing the same arguments gives the same configuration.
from careamics.config import create_advanced_care_config

# Both metrics below are scale invariant: the restored prediction, the low-SNR input and
# the high-SNR ground truth are not guaranteed to share an intensity scale, and a plain
# PSNR or SSIM would largely measure that difference rather than image quality.
# `scale_invariant_psnr` is not re-exported by `careamics.metrics`, so it has to be
# imported from the submodule. `microssim` is the SSIM implementation CAREamics itself
# depends on, and it is installed alongside CAREamics.
from careamics.metrics.metrics import scale_invariant_psnr
from microssim import micro_structural_similarity

TIFF_SUFFIXES = {".tif", ".tiff"}


def require_path(path_value, field_name, must_exist=True):
    """Return `path_value` as a Path, with a clear error if it is unset or missing.

    An empty string would otherwise become `Path(".")`, which silently points at the
    whole Colab working directory.
    """
    path_text = str(path_value).strip()
    if not path_text:
        raise ValueError(f"Please set `{field_name}` before running this cell.")

    path = Path(path_text).expanduser()
    if must_exist and not path.exists():
        raise FileNotFoundError(f"`{field_name}` does not exist: {path}")
    return path


print(f"CAREamics version: {careamics.__version__}")


# **2. Initialise the Colab session**
---

## **2.1. Check for GPU access**

In [ ]:
#@markdown ##Run this cell to check if you have GPU access
import torch

if torch.cuda.is_available():
    print("You have GPU access.")
    print(torch.cuda.get_device_name(0))
else:
    print("You do NOT have GPU access.")
    print("Go to 'Runtime -> Change runtime type' and select a GPU hardware accelerator,")
    print("then re-run the notebook. Expect slow performance on CPU.")

## **2.2. Mount your Google Drive**

In [ ]:
#@markdown ##Play the cell to connect your Google Drive to Colab
#@markdown * Follow the instructions.
#@markdown * Click on "Files" on the left. Refresh it — your Google Drive appears as "drive".

from google.colab import drive
drive.mount('/content/gdrive')

# **3. Select your parameters and paths**
---

## **3.1. Setting the main training parameters**
<font size = 4>`Training_source` (low SNR) and `Training_target` (high SNR) should each point to a folder of 3D `.tif` or `.tiff` stacks, or to a single TIFF stack. Paired source and target files must have **matching file names**.

<font size = 4>`model_path` is where the trained model, its checkpoints and the quality control results are written. Point it at a folder **on your Google Drive**, otherwise everything is lost when the Colab runtime shuts down.

<font size = 4>**Patch size** is set separately in `Z` and in `XY`, because 3D microscopy stacks are usually anisotropic — far fewer slices than pixels within a slice. Each value must be a **power of 2 and at least 8** (8, 16, 32, 64 ...) and must not exceed the corresponding dimension of your stacks. If your stacks have very few slices, use `patch_size_z = 8`.

<font size = 4>**Batch size** is the number of patches seen per training step. 3D patches are much larger than 2D ones, so the default is far smaller than in the 2D notebook; lower it further if you run out of GPU memory.

<font size = 4>**Prediction tile size** is used in sections 5.2 and 6.1. Large stacks are restored tile by tile to keep memory use low, and is again set separately in `Z` and `XY`. Tiles overlap by 8 slices in `Z` and 48 pixels in `XY`, so the tile size must be comfortably larger than that: keep `Z` at 16 or more and `XY` at 128 or more. Reduce them if you run out of GPU memory.

<font size = 4>**Reproducibility.** Training is random in several ways: which patches are drawn from your images, how each one is augmented. By default CAREamics draws a new random seed every run, so the same data and settings give a slightly different model each time. Leave **`Use_fixed_seed`** enabled to pin the seed, which makes a run repeatable and is what you want when comparing settings. Change `seed` to see how much run-to-run variation your data produces, or disable `Use_fixed_seed` to let CAREamics pick one at random.

<font size = 4>Two caveats: on a GPU a few operations remain nondeterministic, so results can still differ in the last decimal places; and a pre-trained model loaded in section 3.3 keeps the seed stored in its own checkpoint.


In [ ]:
#@markdown ###Path to the low-SNR input stacks (folder of 3D .tif/.tiff stacks, or a single file):
Training_source = "" #@param {type:"string"}
#@markdown ###Path to the matching high-SNR target stacks (folder of 3D .tif/.tiff stacks, or a single file):
Training_target = "" #@param {type:"string"}

#@markdown ###Model name and output folder (use a folder on your Google Drive):
model_name = "my_care_3d_model" #@param {type:"string"}
model_path = "" #@param {type:"string"}

#@markdown ###Training parameters
#@markdown Number of epochs:
number_of_epochs = 100 #@param {type:"number"}
#@markdown Patch size within a slice (pixels, square, a power of 2 and at least 8):
patch_size_xy = 64 #@param {type:"number"}
#@markdown Patch size across slices (a power of 2 and at least 8):
patch_size_z = 16 #@param {type:"number"}
#@markdown Batch size:
batch_size = 8 #@param {type:"number"}
#@markdown Number of patches held out for validation:
n_val_patches = 8 #@param {type:"number"}

#@markdown ###Prediction parameters
#@markdown Tile size within a slice, used in sections 5.2 and 6.1 (pixels, square, at least 128):
prediction_tile_size_xy = 128 #@param {type:"number"}
#@markdown Tile size across slices (at least 16):
prediction_tile_size_z = 32 #@param {type:"number"}

# CAREamics needs one overlap value per axis in 3D; its 2D default of (48, 48) does
# not apply here. These are the values used by the CAREamics 3D examples, and
# they set the minimum usable tile sizes quoted above.
prediction_tile_overlap = (8, 48, 48)
prediction_tile_size = (
    int(prediction_tile_size_z),
    int(prediction_tile_size_xy),
    int(prediction_tile_size_xy),
)

#@markdown ###Reproducibility
#@markdown Use a fixed random seed, so that the same data and settings reproduce the same result:
Use_fixed_seed = True #@param {type:"boolean"}
#@markdown Seed value:
seed = 42 #@param {type:"number"}


## **3.2. Data augmentation**
<font size = 4>Data augmentation (flips and 90° rotations) usually improves results and is recommended.

In [ ]:
#@markdown ##Enable or disable data augmentation:
Use_Data_augmentation = True #@param {type:"boolean"}

## **3.3. Using a pre-trained model**
<font size = 4>You can continue training from a previously trained CAREamics model. Provide the path to a checkpoint (`.ckpt`). The pre-trained model's configuration is reused, so the parameters above are ignored when this is enabled.

In [ ]:
#@markdown ##Load weights from a pre-trained CAREamics model
Use_pretrained_model = False #@param {type:"boolean"}
#@markdown ###If enabled, provide the path to the checkpoint (.ckpt) file:
pretrained_model_path = "" #@param {type:"string"}

# **4. Train the network**
---

## **4.1. Prepare the training data and model**

In [ ]:
#@markdown ##Create the configuration and the CAREamist
# Augmentations: None -> default (flips + 90-degree rotations); [] -> disabled
augmentations = None if Use_Data_augmentation else []

work_dir = require_path(model_path, "model_path", must_exist=False)
work_dir.mkdir(parents=True, exist_ok=True)

if Use_pretrained_model:
    checkpoint_path = require_path(pretrained_model_path, "pretrained_model_path")
    print(f"Loading pre-trained model from: {checkpoint_path}")
    careamist = CAREamist(checkpoint_path=checkpoint_path, work_dir=work_dir)
else:
    config = create_advanced_care_config(
        experiment_name=model_name,
        data_type="tiff",
        axes="ZYX",
        patch_size=(patch_size_z, patch_size_xy, patch_size_xy),
        batch_size=batch_size,
        num_epochs=number_of_epochs,
        n_val_patches=n_val_patches,
        augmentations=augmentations,
        seed=int(seed) if Use_fixed_seed else None,
    )
    print(config)
    careamist = CAREamist(config, work_dir=work_dir)


## **4.2. Start training**
<font size = 4>Training checkpoints are saved automatically to your output folder. If you loaded a pre-trained model and only want to run predictions, leave the training paths empty and skip this section.


In [ ]:
#@markdown ##Start training
training_source_text = str(Training_source).strip()
training_target_text = str(Training_target).strip()

if not training_source_text or not training_target_text:
    if Use_pretrained_model:
        print("No training paths provided. Keeping the loaded model for evaluation/prediction.")
    else:
        missing = []
        if not training_source_text:
            missing.append("Training_source")
        if not training_target_text:
            missing.append("Training_target")
        raise ValueError("Please set " + " and ".join(missing) + " before training.")
else:
    careamist.train(
        train_data=require_path(Training_source, "Training_source"),
        train_data_target=require_path(Training_target, "Training_target"),
    )
    print("Training complete.")


# **5. Evaluate your model**
---

## **5.1. Inspection of the loss function**
---
<font size = 4>Comparing the training loss with the validation loss shows how training progressed. **Training loss** is the error on the patches the network learned from; **validation loss** is the same error on a small set of patches held back from training, so it indicates how well the network does on data it has not been trained on.

<font size = 4>**What a healthy run looks like:** both curves drop quickly and then flatten. Once they are flat, the network has converged and further training will not help. If both are still clearly decreasing at the last epoch, increase `number_of_epochs`. If the validation loss starts *rising* while the training loss keeps falling, the network is overfitting, and you should reduce `number_of_epochs` or provide more training data.

<font size = 4>Because CARE is supervised against a genuine high-SNR target, its loss usually falls further and more visibly than the self-supervised losses of Noise2Void or Noise2Noise. How far it can fall is still limited by how much of the high-SNR image is actually predictable from the low-SNR one, so it will not reach zero.

<font size = 4>**One thing that looks wrong but is normal:** the validation loss can sit above or below the training loss, and moves between runs.** It is measured on only `n_val_patches` patches (8 by default), chosen at random each time you train. Its absolute value is therefore a noisy estimate. Read the *shape* of the curve rather than the number, and raise `n_val_patches` if you want a steadier estimate.

<font size = 4>The loss curves tell you whether training converged. Whether the model actually improved your images is answered in **section 5.2**.

In [ ]:
#@markdown ##Plot the training and validation loss vs. epoch
try:
    loss_dict = careamist.get_losses()
except Exception as error:
    raise RuntimeError(
        "Could not read training losses. This section is only available after "
        "training in the current work_dir."
    ) from error

if not loss_dict.get("train_loss"):
    raise RuntimeError("No training loss values were found in the CSV logs.")

plt.figure(figsize=(8, 5))
plt.plot(loss_dict["train_epoch"], loss_dict["train_loss"], label="Train loss")
if loss_dict.get("val_loss"):
    plt.plot(loss_dict["val_epoch"], loss_dict["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.title("Training losses")
plt.show()


## **5.2. Error mapping and quality metrics estimation**
---
<font size = 4>This section evaluates the trained model on a **Quality Control dataset**: pairs of low-SNR stacks (`Source_QC_folder`) and matching high-SNR stacks (`Target_QC_folder`). Paired files must have **the same file name**, and these fields of view should be **held back from training**.

<font size = 4>Each metric is computed twice: once for the **prediction against the ground truth**, and once for the **low-SNR input against the ground truth**. The second is a baseline. If denoising helped, the prediction scores better than the input, and the gap between the two is the improvement the model gave you.

<font size = 4>**SI-PSNR (scale-invariant peak signal-to-noise ratio)**, in decibels: how closely an image matches the ground truth. Higher is better, and roughly +3 means the error was halved.

<font size = 4>**MicroSSIM**: a structural similarity index adapted to microscopy. It asks whether the same *structures* are present rather than whether pixel values match, and runs from 0 to 1, where 1 is a perfect match. MicroSSIM is a 2D measure, so it is computed slice by slice and averaged over the stack, while SI-PSNR is computed over the whole volume at once.

<font size = 4>Both are **scale invariant**, so they are unaffected by the prediction, the low-SNR acquisition and the high-SNR acquisition sitting on different intensity scales. This is the same metric used in the [CAREamics CARE example](https://github.com/CAREamics/careamics-examples/blob/main/applications/care/denoising/2D/U2OS_CARE.ipynb).

<font size = 4>The figure shows one `Z` slice of the low-SNR input, the prediction and the ground truth, with a zoomed-in crop of each underneath so you can judge fine detail. The MicroSSIM error map is saved as a 3D stack, one map per slice. Per-image scores, and a MicroSSIM error map for each image, are written to a `Quality Control` folder next to your model, along with `QC_metrics_<model_name>.csv`.

In [ ]:
#@markdown ##Provide the Quality Control folders (paired low-SNR and high-SNR 3D .tif/.tiff stacks)
Source_QC_folder = "" #@param {type:"string"}
Target_QC_folder = "" #@param {type:"string"}
#@markdown Size of the zoomed-in crop shown under the full images (pixels, square):
qc_crop_size = 256 #@param {type:"number"}
#@markdown Z slice to display (-1 shows the middle slice of the stack):
qc_z_slice = -1 #@param {type:"number"}

source_qc_path = require_path(Source_QC_folder, "Source_QC_folder")
target_qc_path = require_path(Target_QC_folder, "Target_QC_folder")

# ZeroCostDL4Mic convention: QC results live in a "Quality Control" folder next to the
# model they describe.
qc_dir = work_dir / model_name / "Quality Control"
qc_dir.mkdir(parents=True, exist_ok=True)

# Let CAREamics list the source files itself, so that each prediction stays paired with
# the file it came from.
qc_predictions, qc_sources = careamist.predict(
    pred_data=source_qc_path,
    tile_size=prediction_tile_size,
    tile_overlap=prediction_tile_overlap,
)

csv_path = qc_dir / f"QC_metrics_{model_name}.csv"
scores = {"si_psnr_pred": [], "si_psnr_input": [], "ssim_pred": [], "ssim_input": []}
first_image = None  # kept for the figure; the rest are not held in memory

with open(csv_path, "w", newline="") as csv_file:
    writer = csv.writer(csv_file)
    writer.writerow([
        "image",
        "Prediction v. GT SI-PSNR",
        "Input v. GT SI-PSNR",
        "Prediction v. GT MicroSSIM",
        "Input v. GT MicroSSIM",
    ])

    for prediction, source in zip(qc_predictions, qc_sources):
        source_file = Path(source)
        target_file = target_qc_path / source_file.name
        if not target_file.exists():
            raise FileNotFoundError(
                f"No ground truth named '{source_file.name}' in {target_qc_path}. "
                "Paired QC images must have matching file names."
            )

        gt = tifffile.imread(target_file).astype(np.float32)
        noisy = tifffile.imread(source_file).astype(np.float32)
        denoised = np.asarray(prediction).squeeze().astype(np.float32)

        # SI-PSNR is computed over the whole volume. MicroSSIM is a 2D measure:
        # given a stack it returns one score per slice, which we average.
        si_psnr_pred = scale_invariant_psnr(gt, denoised)
        si_psnr_input = scale_invariant_psnr(gt, noisy)
        ssim_pred = float(np.mean(micro_structural_similarity(gt, denoised)))
        ssim_input = float(np.mean(micro_structural_similarity(gt, noisy)))

        writer.writerow([source_file.name, si_psnr_pred, si_psnr_input,
                         ssim_pred, ssim_input])
        scores["si_psnr_pred"].append(si_psnr_pred)
        scores["si_psnr_input"].append(si_psnr_input)
        scores["ssim_pred"].append(ssim_pred)
        scores["ssim_input"].append(ssim_input)

        # Per-pixel MicroSSIM error map, saved for inspection in Fiji or napari.
        # The components have to be requested slice by slice: on a whole stack
        # microssim returns a list of results rather than a single one.
        error_map = np.stack([
            np.asarray(
                micro_structural_similarity(
                    gt[z], denoised[z], return_individual_components=True
                ).SSIM
            )
            for z in range(gt.shape[0])
        ]).astype(np.float32)
        tifffile.imwrite(qc_dir / f"MicroSSIM_GTvsPrediction_{source_file.name}", error_map)

        if first_image is None:
            first_image = (noisy, denoised, gt, si_psnr_input, si_psnr_pred)

        print(f"{source_file.name}: SI-PSNR {si_psnr_input:.2f} -> {si_psnr_pred:.2f}, "
              f"MicroSSIM {ssim_input:.4f} -> {ssim_pred:.4f}  (input -> prediction)")

label = "Mean over QC images" if len(scores["si_psnr_pred"]) > 1 else "QC image"
print(f"\n{'':32}{'input':>10}{'prediction':>13}{'gain':>10}")
print(f"{label + ', SI-PSNR:':32}"
      f"{np.mean(scores['si_psnr_input']):>10.2f}{np.mean(scores['si_psnr_pred']):>13.2f}"
      f"{np.mean(scores['si_psnr_pred']) - np.mean(scores['si_psnr_input']):>+10.2f}")
print(f"{label + ', MicroSSIM:':32}"
      f"{np.mean(scores['ssim_input']):>10.4f}{np.mean(scores['ssim_pred']):>13.4f}"
      f"{np.mean(scores['ssim_pred']) - np.mean(scores['ssim_input']):>+10.4f}")
print(f"\nMetrics and error maps saved to: {qc_dir}")

# Display one slice of the first QC stack: full field of view on top, a zoomed-in
# crop underneath.
noisy, denoised, gt, si_psnr_input, si_psnr_pred = first_image
z = gt.shape[0] // 2 if int(qc_z_slice) < 0 else int(qc_z_slice)
if z >= gt.shape[0]:
    raise IndexError(
        f"qc_z_slice must be between 0 and {gt.shape[0] - 1} for this stack."
    )
noisy, denoised, gt = noisy[z], denoised[z], gt[z]
height, width = gt.shape[-2:]
half = min(int(qc_crop_size), height, width) // 2
rows = slice(height // 2 - half, height // 2 + half)
cols = slice(width // 2 - half, width // 2 + half)

panels = [
    (noisy, f"Low-SNR input\nSI-PSNR: {si_psnr_input:.2f}"),
    (denoised, f"Prediction\nSI-PSNR: {si_psnr_pred:.2f}"),
    (gt, "Ground truth"),
]
print(f"\nShowing Z slice {z}.")
fig, ax = plt.subplots(2, 3, figsize=(15, 10))
for column, (image, title) in enumerate(panels):
    ax[0, column].imshow(image, cmap="gray")
    ax[0, column].set_title(title)
    ax[1, column].imshow(image[rows, cols], cmap="gray")
for axis in ax.ravel():
    axis.axis("off")
plt.show()


# **6. Using the trained model**
---

## **6.1. Generate predictions from an unseen dataset**

In [ ]:
#@markdown ###Path to the data to denoise and the folder where results are saved:
Data_folder = "" #@param {type:"string"}
Result_folder = "" #@param {type:"string"}

data_path = require_path(Data_folder, "Data_folder")
result_dir = require_path(Result_folder, "Result_folder", must_exist=False)
result_dir.mkdir(parents=True, exist_ok=True)

predictions, sources = careamist.predict(
    pred_data=data_path,
    tile_size=prediction_tile_size,
    tile_overlap=prediction_tile_overlap,
)

for pred, source in zip(predictions, sources):
    out_name = Path(source).stem + "_denoised.tif"
    out_path = result_dir / out_name
    tifffile.imwrite(out_path, np.asarray(pred).squeeze().astype(np.float32))
    print(f"Saved: {out_path}")


## **6.2. Assess the predicted output**

In [ ]:
#@markdown ##Display one slice of an input stack next to its denoised prediction
idx = 0 #@param {type:"number"}
#@markdown Z slice to display (-1 shows the middle slice of the stack):
z_slice = -1 #@param {type:"number"}
idx = int(idx)

if not predictions:
    raise RuntimeError("No predictions are available. Run section 6.1 first.")
if idx < 0 or idx >= len(predictions):
    raise IndexError(f"idx must be between 0 and {len(predictions) - 1}.")

# `sources` comes from CAREamics itself, so the input always matches the prediction.
input_img = tifffile.imread(sources[idx])
pred_img = np.asarray(predictions[idx]).squeeze()

z = input_img.shape[0] // 2 if int(z_slice) < 0 else int(z_slice)
if z >= input_img.shape[0]:
    raise IndexError(
        f"z_slice must be between 0 and {input_img.shape[0] - 1} for this stack."
    )
input_img, pred_img = input_img[z], pred_img[z]

fig, ax = plt.subplots(1, 2, figsize=(12, 6))
ax[0].imshow(input_img, cmap="gray")
ax[0].set_title("Input (low SNR)")
ax[1].imshow(pred_img, cmap="gray")
ax[1].set_title("Prediction (restored)")
for axis in ax:
    axis.axis("off")
fig.suptitle(f"{Path(sources[idx]).name} — Z slice {z}")
plt.show()


## **6.3. Download your predictions**
---

<font size = 4>**Store your data** and ALL its results elsewhere by downloading them from your Google Drive, and then clean up the original folder tree (dataset, results, trained model) if you plan to train or use another network. Please note that the notebook will otherwise **OVERWRITE** all files which have the same name.

# **7. Version log**
---
<font size = 4>**v1.0 (CAREamics)**:
*   First release of the CAREamics-powered CARE 3D notebook. It runs alongside the existing CSBDeep `CARE_3D_ZeroCostDL4Mic` notebook, which is unchanged.
*   Built on CAREamics 0.3.2 (`create_advanced_care_config` + `CAREamist`) with `axes="ZYX"`.
*   Patch and prediction tile sizes are set separately in `Z` and `XY`, and prediction passes an explicit per-axis tile overlap.
*   Section 5.2 reports scale-invariant PSNR over the volume and MicroSSIM averaged over slices, for the prediction and for the low-SNR input against the ground truth, saves MicroSSIM error map stacks, and writes `QC_metrics_<model_name>.csv`.
*   Optional fixed random seed (section 3.1) for reproducible training and prediction.


# **Thank you for using CARE 3D!**